In [1]:
import os
import time
import copy

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath(".."))
from models import *

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), normalize]))
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

STEP1_PATH = 'result/vgg16/model_best.pth.tar'   # trained VGG16 from HW3 (no quantization)
save_root = 'result'                             # fine-tuned models go to result/vgg16_gamma_<g>/


def load_step1():
    """Fresh copy of the step 1 model. Every run starts from here."""
    model = VGG16().to(device)
    model.load_state_dict(torch.load(STEP1_PATH, map_location=device)['state_dict'])
    return model


def conv1_abs_sum(model):
    """Absolute sum of the first conv layer's weights (features[0] in VGG16)."""
    return model.features[0].weight.abs().sum().item()


def test_accuracy(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            correct += (model(data).argmax(dim=1) == target).sum().item()
    return 100. * correct / len(testloader.dataset)

Using device: mps


In [2]:
model = load_step1()
print(model.features[0])   # first conv layer: the one loss2 penalizes

step1_acc = test_accuracy(model)
step1_sum = conv1_abs_sum(model)
print(f'Step 1  test accuracy: {step1_acc:.2f}%   conv1 weight abs sum: {step1_sum:.4f}')

Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Step 1  test accuracy: 92.51%   conv1 weight abs sum: 318.5653


In [3]:
# ---- fixed training settings, shared by every gamma ----
FT_LR = 0.01
FT_MOMENTUM = 0.9
FT_WEIGHT_DECAY = 1e-4
FT_EPOCHS = 10


def gamma_dir(gamma):
    return os.path.join(save_root, f'vgg16_gamma_{gamma:g}')


def finetune(gamma):
    """Fine-tune the step 1 model with loss = loss1 + gamma * loss2 and save the final model."""
    torch.manual_seed(0)                       # same data order for every gamma
    model = load_step1()                       # always start from step 1
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=FT_LR, momentum=FT_MOMENTUM,
                                weight_decay=FT_WEIGHT_DECAY)   # fresh optimizer: no momentum carried over

    for epoch in range(FT_EPOCHS):
        model.train()
        sum_loss1 = sum_loss2 = 0.0
        start = time.time()
        for data, target in trainloader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)

            loss1 = criterion(output, target)
            loss2 = model.features[0].weight.abs().sum()   # first conv layer only; kept as a tensor so it backprops
            loss = loss1 + gamma * loss2

            loss.backward()
            optimizer.step()
            sum_loss1 += loss1.item() * data.size(0)
            sum_loss2 += loss2.item() * data.size(0)

        n = len(trainloader.dataset)
        print(f'gamma={gamma:g}  epoch {epoch+1}/{FT_EPOCHS}  loss1 {sum_loss1/n:.4f}  loss2 {sum_loss2/n:.2f}  '
              f'test acc {test_accuracy(model):.2f}%  ({time.time()-start:.0f}s)')

    acc, wsum = test_accuracy(model), conv1_abs_sum(model)
    os.makedirs(gamma_dir(gamma), exist_ok=True)
    torch.save({'gamma': gamma, 'state_dict': model.state_dict(), 'test_acc': acc, 'conv1_abs_sum': wsum,
                'settings': dict(lr=FT_LR, momentum=FT_MOMENTUM, weight_decay=FT_WEIGHT_DECAY,
                                 epochs=FT_EPOCHS, batch_size=batch_size)},
               os.path.join(gamma_dir(gamma), 'model.pth'))
    print(f'==> gamma={gamma:g}: test accuracy {acc:.2f}%, conv1 abs sum {wsum:.4f}')
    return model

After adding the new loss term (γ = 1)

`loss = loss1 + loss2`. Training log: clear this cell's output before exporting the PDF.

In [6]:
finetune(gamma=1)

gamma=1  epoch 1/10  loss1 0.7628  loss2 32.13  test acc 41.08%  (66s)
gamma=1  epoch 2/10  loss1 0.5213  loss2 22.89  test acc 50.04%  (65s)
gamma=1  epoch 3/10  loss1 0.4353  loss2 21.98  test acc 60.71%  (65s)
gamma=1  epoch 4/10  loss1 0.4123  loss2 22.23  test acc 70.58%  (66s)
gamma=1  epoch 5/10  loss1 0.3809  loss2 21.68  test acc 54.12%  (66s)
gamma=1  epoch 6/10  loss1 0.3822  loss2 21.55  test acc 70.64%  (66s)
gamma=1  epoch 7/10  loss1 0.3294  loss2 21.37  test acc 81.95%  (66s)
gamma=1  epoch 8/10  loss1 0.3326  loss2 21.29  test acc 44.46%  (66s)
gamma=1  epoch 9/10  loss1 0.3022  loss2 21.17  test acc 15.52%  (66s)
gamma=1  epoch 10/10  loss1 0.3067  loss2 21.57  test acc 74.37%  (66s)
==> gamma=1: test accuracy 74.37%, conv1 abs sum 20.6638


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2

In [7]:
gammas = [1e-4, 1e-3, 1e-2, 1e-1]

for g in gammas:
    finetune(gamma=g)

gamma=0.0001  epoch 1/10  loss1 0.0362  loss2 314.88  test acc 92.60%  (67s)
gamma=0.0001  epoch 2/10  loss1 0.0294  loss2 307.35  test acc 92.69%  (66s)
gamma=0.0001  epoch 3/10  loss1 0.0246  loss2 300.19  test acc 92.81%  (65s)
gamma=0.0001  epoch 4/10  loss1 0.0204  loss2 293.29  test acc 92.86%  (66s)
gamma=0.0001  epoch 5/10  loss1 0.0169  loss2 286.46  test acc 92.78%  (66s)
gamma=0.0001  epoch 6/10  loss1 0.0160  loss2 280.01  test acc 92.83%  (66s)
gamma=0.0001  epoch 7/10  loss1 0.0144  loss2 273.78  test acc 92.82%  (66s)
gamma=0.0001  epoch 8/10  loss1 0.0132  loss2 267.79  test acc 92.68%  (65s)
gamma=0.0001  epoch 9/10  loss1 0.0117  loss2 262.06  test acc 92.72%  (67s)
gamma=0.0001  epoch 10/10  loss1 0.0120  loss2 256.38  test acc 92.93%  (66s)
==> gamma=0.0001: test accuracy 92.93%, conv1 abs sum 253.4615
gamma=0.001  epoch 1/10  loss1 0.0363  loss2 289.97  test acc 92.58%  (66s)
gamma=0.001  epoch 2/10  loss1 0.0297  loss2 242.38  test acc 92.68%  (66s)
gamma=0.001  e

In [8]:
all_gammas = [1, 1e-4, 1e-3, 1e-2, 1e-1]

results = [{'gamma': 0, 'acc': step1_acc, 'sum': step1_sum}]   # step 1 = gamma 0
for g in sorted(all_gammas):
    m = VGG16().to(device)
    m.load_state_dict(torch.load(os.path.join(gamma_dir(g), 'model.pth'), map_location=device)['state_dict'])
    results.append({'gamma': g, 'acc': test_accuracy(m), 'sum': conv1_abs_sum(m)})

print(f'{"gamma":>8} | {"test acc (%)":>12} | {"conv1 abs sum":>13} | {"vs step 1":>9}')
print('-' * 52)
for r in results:
    print(f'{r["gamma"]:>8g} | {r["acc"]:>12.2f} | {r["sum"]:>13.4f} | {100 * r["sum"] / step1_sum:>8.1f}%')

   gamma | test acc (%) | conv1 abs sum | vs step 1
----------------------------------------------------
       0 |        92.51 |      318.5653 |    100.0%
  0.0001 |        92.93 |      253.4615 |     79.6%
   0.001 |        92.67 |       79.5154 |     25.0%
    0.01 |        88.59 |       14.5317 |      4.6%
     0.1 |        78.31 |        6.4619 |      2.0%
       1 |        74.37 |       20.6638 |      6.5%


Optimal γ

Criterion: test accuracy **above 80%**, and the smallest conv1 weight absolute sum among those.

In [9]:
candidates = [r for r in results if r['gamma'] > 0 and r['acc'] > 80.0]
best = min(candidates, key=lambda r: r['sum'])
gamma_opt = best['gamma']

print(f'Optimal gamma = {gamma_opt:g}')
print(f'Cost function: loss = CrossEntropyLoss(output, target) + {gamma_opt:g} * model.features[0].weight.abs().sum()')
print(f'Test accuracy = {best["acc"]:.2f}%   conv1 weight abs sum = {best["sum"]:.4f}')

Optimal gamma = 0.01
Cost function: loss = CrossEntropyLoss(output, target) + 0.01 * model.features[0].weight.abs().sum()
Test accuracy = 88.59%   conv1 weight abs sum = 14.5317


In [ ]:
g1 = next(r for r in results if r['gamma'] == 1)
rows = [
    ('1. Initial version',        'loss1',                              step1_acc,   step1_sum),
    ('2. New loss term (γ = 1)',  'loss1 + loss2',                      g1['acc'],   g1['sum']),
    ('3. Fine-tuned (optimal γ)', f'loss1 + γ × loss2 (γ = {gamma_opt:g})', best['acc'], best['sum']),
]
print(f'{"Case":<27} | {"Cost function":<28} | {"Test acc (%)":>12} | {"Conv1 abs sum":>13}')
print('-' * 90)
for case, cost, acc, s in rows:
    print(f'{case:<27} | {cost:<28} | {acc:>12.2f} | {s:>13.4f}')

Case                        | Cost function                | Test acc (%) | Conv1 abs sum
------------------------------------------------------------------------------------------
1. Initial version          | loss1                        |        92.51 |      318.5653
2. New loss term (γ = 1)    | loss1 + loss2                |        74.37 |       20.6638
3. Fine-tuned (optimal γ)   | loss1 + γ × loss2 (γ = 0.01) |        88.59 |       14.5317


**Discussion**

| γ | Test acc (%) | Conv1 abs sum | % of initial |
|---|---|---|---|
| 0 (initial) | 92.51 | 318.57 | 100% |
| 1e-4 | 92.93 | 253.46 | 79.6% |
| 1e-3 | 92.67 | 79.52 | 25.0% |
| **1e-2 (optimal)** | **88.59** | **14.53** | **4.6%** |
| 1e-1 | 78.31 | 6.46 | 2.0% |
| 1 | 74.37 | 20.66 | 6.5% |

- **Trend:** as γ increases, the conv1 weight absolute sum drops (loss2 pushes every weight toward 0 by lr·γ per step) …
- **Why the weights can shrink with little accuracy loss:** conv1 is followed by BatchNorm, which normalizes each output channel and cancels a uniform scaling of the conv weights, so shrinking them barely changes the network's function.
- **Large γ (e.g. 1):** each step moves every weight by about lr·γ, which is larger than most weights themselves, so they oscillate around 0 instead of settling. That noise can hurt accuracy and can leave a larger final sum than a moderate γ.